In [1]:
MODEL="embeddinggemma-2:270m"

import sys
from pathlib import Path

import numpy as np
from rich.console import Console
from rich.table import Table

# Ensure project root and src/ are on sys.path so `core` and `service` import cleanly
PROJECT_ROOT = Path("..").resolve()
SRC_DIR = PROJECT_ROOT / "src"
for _p in (SRC_DIR, PROJECT_ROOT):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

from gosmart_graphite.core.config import Settings
from gosmart_graphite.model.llm_config import LLM_Config
from gosmart_graphite.service.embedding_service import EmbeddingService, SimilarityMetric
from gosmart_graphite.service.llm_config_service import llm_config_service

console = Console()
env_file = PROJECT_ROOT / ".env"
settings = Settings(_env_file=str(env_file) if env_file.exists() else None)
llm_cfg = await llm_config_service.resolve_config_for_group(tag="embedding")
if llm_cfg is None:
    llm_cfg = LLM_Config(
        tag="embedding-v1",
        provider="ollama",
        model=MODEL,
        url="http://localhost:11434",
        dimensions=768,
    )
vector_service = EmbeddingService(config=llm_cfg, settings=settings)

print(f"Provider   : {vector_service.resolve_provider()}")
print(f"Embedder   : {vector_service.resolve_embedder_name()}")
print(f"Server URL : {llm_cfg.url or '-'}")
print(f"Dimensions : {llm_cfg.dimensions}")


Provider   : ollama
Embedder   : ollama/embeddinggemma-2:270m
Server URL : http://localhost:11434
Dimensions : 768


## 1. Edit Query & 12 Target Strings

Modify `query_text` or any of the 12 strings in `target_texts` below and re-run the cells to explore how the embedding vectors and similarity scores behave.


In [2]:
# Query string to search against the 12 target strings (edit freely!)
query_text = "Patient presents with acute onset shortness of breath and denies chest pain."
# query_text = "Patient denies having any acute onset shortness of breath and denies chest pain at all."
# query_text = "Patient reports frequent urination episodes for the past week"


# 12 Target strings spanning closely related cardiopulmonary notes, negation differences, and unrelated clinical/non-clinical topics
target_texts = [
    "Patient presents with acute onset shortness of breath and denies chest pain.",  # 0: Exact match
    "Patient reports sudden dyspnea without any chest discomfort or angina.",        # 1: High semantic similarity (synonym)
    "Emergency visit for severe chest pain radiating to left arm with diaphoresis.", # 2: Related domain, opposite chest pain finding
    "Chest CT angiogram demonstrates bilateral pulmonary embolism in lower lobes.",  # 3: Related cardiopulmonary imaging finding
    "Chest radiograph shows clear lungs bilaterally with normal cardiac silhouette.", # 4: Normal chest imaging report
    "Brain MRI T1 sagittal reveals no acute intracranial hemorrhage or mass effect.", # 5: Different imaging modality/organ
    "Follow-up for Type 2 diabetes mellitus; HbA1c is 6.8% on metformin therapy.",   # 6: Unrelated internal medicine note
    "Right knee arthroscopy performed for medial meniscus tear rehabilitation.",     # 7: Orthopedics / surgical procedure
    "Routine pediatric well-child visit and scheduled childhood immunizations.",     # 8: Pediatrics routine care
    "DICOM C-STORE transfer syntax negotiation over TCP port 11112 succeeded.",
    "Patient complains of frequent night time urination.",
    "Patient complains of frequent daytime urination but not at bed time.",
]

print(f"Query : {query_text}")
print(f"Targets count: {len(target_texts)}")
for idx, text in enumerate(target_texts):
    print(f"  [{idx}] {text}")


Query : Patient presents with acute onset shortness of breath and denies chest pain.
Targets count: 12
  [0] Patient presents with acute onset shortness of breath and denies chest pain.
  [1] Patient reports sudden dyspnea without any chest discomfort or angina.
  [2] Emergency visit for severe chest pain radiating to left arm with diaphoresis.
  [3] Chest CT angiogram demonstrates bilateral pulmonary embolism in lower lobes.
  [4] Chest radiograph shows clear lungs bilaterally with normal cardiac silhouette.
  [5] Brain MRI T1 sagittal reveals no acute intracranial hemorrhage or mass effect.
  [6] Follow-up for Type 2 diabetes mellitus; HbA1c is 6.8% on metformin therapy.
  [7] Right knee arthroscopy performed for medial meniscus tear rehabilitation.
  [8] Routine pediatric well-child visit and scheduled childhood immunizations.
  [9] DICOM C-STORE transfer syntax negotiation over TCP port 11112 succeeded.
  [10] Patient complains of frequent night time urination.
  [11] Patient compl

## 2. Generate L2-Normalized Vector Embeddings

Compute the normalized embedding vector for `query_text` and each of the 10 `target_texts` using `vector_service.embed_text()`.


In [3]:
query_vector = await vector_service.embed_text_async(query_text)
target_vectors = [await vector_service.embed_text_async(t) for t in target_texts]

print(f"Query vector dimension : {len(query_vector)}, L2 norm: {np.linalg.norm(query_vector):.6f}")
print(f"Target vectors count   : {len(target_vectors)}, each of dimension {len(target_vectors[0])}")
print(f"Sample query_vector[:5]: {[round(v, 5) for v in query_vector[:5]]}")


2026-11-21 10:48:47 [info     ] Embedding text: title: none | text: Patient presents with acute onset shortness of breath and denies chest pain.
2026-11-21 10:48:47 [debug    ] initializing_genkit_ollama     base_url=http://localhost:11434 tag=embedding-v1
2026-11-21 10:48:47 [info     ] Embedding text: title: none | text: Patient presents with acute onset shortness of breath and denies chest pain.
2026-11-21 10:48:47 [info     ] Embedding text: title: none | text: Patient reports sudden dyspnea without any chest discomfort or angina.
2026-11-21 10:48:47 [info     ] Embedding text: title: none | text: Emergency visit for severe chest pain radiating to left arm with diaphoresis.
2026-11-21 10:48:47 [info     ] Embedding text: title: none | text: Chest CT angiogram demonstrates bilateral pulmonary embolism in lower lobes.
2026-11-21 10:48:47 [info     ] Embedding text: title: none | text: Chest radiograph shows clear lungs bilaterally with normal cardiac silhouette.
2026-11-21 10:48:47 [

## 3. Compare Similarity Algorithms (`[0.0, 1.0]` Scores in Index Order)

We compare `query_vector` against `target_vectors` across the four algorithms in `embedding_service.py`:
- **Cosine Similarity (`cosine`)**: $\frac{\cos(\theta) + 1}{2}$ (computes full vector norms)
- **Cosine Dot-Product (`dot_product`)**: $\frac{\mathbf{q} \cdot \mathbf{t} + 1}{2}$ (fast inner product on L2-normalized vectors; matches Cosine to floating-point precision)
- **Euclidean Similarity (`euclidean`)**: $\frac{1}{1 + \|\mathbf{q} - \mathbf{t}\|_2}$ (L2 distance mapped to $(0, 1]$)
- **Manhattan Similarity (`manhattan`)**: $\frac{1}{1 + \|\mathbf{q} - \mathbf{t}\|_1}$ (L1 distance across all 768 dimensions mapped to $(0, 1]$)


In [4]:
cosine_scores = vector_service.compare_vectors(query_vector, target_vectors, SimilarityMetric.COSINE)
dot_scores = vector_service.compare_vectors(query_vector, target_vectors, SimilarityMetric.DOT_PRODUCT)
euclidean_scores = vector_service.compare_vectors(query_vector, target_vectors, SimilarityMetric.EUCLIDEAN)
manhattan_scores = vector_service.compare_vectors(query_vector, target_vectors, SimilarityMetric.MANHATTAN)

table = Table(title=f"Similarity Score Comparison (Query: '{query_text}')")
table.add_column("Idx", justify="right", style="cyan", no_wrap=True)
table.add_column("Target String", style="white")
table.add_column("Cosine", justify="right", style="green")
table.add_column("Cosine Dot", justify="right", style="bright_green")
table.add_column("Euclidean (L2)", justify="right", style="yellow")
table.add_column("Manhattan (L1)", justify="right", style="magenta")

for idx, text in enumerate(target_texts):
    table.add_row(
        str(idx),
        text,
        f"{cosine_scores[idx]:.4f}",
        f"{dot_scores[idx]:.4f}",
        f"{euclidean_scores[idx]:.4f}",
        f"{manhattan_scores[idx]:.4f}",
    )

console.print(table)


Similarity Score Comparison (Query: 'Patient presents with acute onset shortness of breath and denies chest pain.')
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┓
┃ Idx ┃ Target String                                     ┃ Cosine ┃ Cosine Dot ┃ Euclidean (L2) ┃ Manhattan (L1) ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━┩
│   0 │ Patient presents with acute onset shortness of    │ 1.0000 │     1.0000 │         1.0000 │         1.0000 │
│     │ breath and denies chest pain.                     │        │            │                │                │
│   1 │ Patient reports sudden dyspnea without any chest  │ 0.9438 │     0.9438 │         0.6784 │         0.0859 │
│     │ discomfort or angina.                             │        │            │                │                │
│   2 │ Emergency visit for severe chest pain radiating   │ 0.9167 │     0.9167 │         0.6340 │         0.0731 │
│     │ to left arm with diaphoresis.                     │        │            │                │                │
│   3 │ Chest CT angiogram demonstrates bilateral         │ 0.8596 │     0.8596 │         0.5716 │         0.0577 │
│     │ pulmonary embolism in lower lobes.                │        │            │                │                │
│   4 │ Chest radiograph shows clear lungs bilaterally    │ 0.8580 │     0.8580 │         0.5702 │         0.0565 │
│     │ with normal cardiac silhouette.                   │        │            │                │                │
│   5 │ Brain MRI T1 sagittal reveals no acute            │ 0.8213 │     0.8213 │         0.5419 │         0.0519 │
│     │ intracranial hemorrhage or mass effect.           │        │            │                │                │
│   6 │ Follow-up for Type 2 diabetes mellitus; HbA1c is  │ 0.8240 │     0.8240 │         0.5438 │         0.0515 │
│     │ 6.8% on metformin therapy.                        │        │            │                │                │
│   7 │ Right knee arthroscopy performed for medial       │ 0.8191 │     0.8191 │         0.5403 │         0.0507 │
│     │ meniscus tear rehabilitation.                     │        │            │                │                │
│   8 │ Routine pediatric well-child visit and scheduled  │ 0.7967 │     0.7967 │         0.5258 │         0.0483 │
│     │ childhood immunizations.                          │        │            │                │                │
│   9 │ DICOM C-STORE transfer syntax negotiation over    │ 0.8222 │     0.8222 │         0.5425 │         0.0512 │
│     │ TCP port 11112 succeeded.                         │        │            │                │                │
│  10 │ Patient complains of frequent night time          │ 0.8641 │     0.8641 │         0.5757 │         0.0577 │
│     │ urination.                                        │        │            │                │                │
│  11 │ Patient complains of frequent daytime urination   │ 0.8721 │     0.8721 │         0.5830 │         0.0597 │
│     │ but not at bed time.                              │        │            │                │                │
└─────┴───────────────────────────────────────────────────┴────────┴────────────┴────────────────┴────────────────┘

## 4. Ranking Order & Algorithm Differences

Compare how each algorithm ranks the 12 target strings from highest to lowest similarity score, and measure the numerical differences between **Cosine / Cosine Dot**, **Euclidean (L2)**, and **Manhattan (L1)**.


In [5]:
# Rank indices from highest to lowest score for each algorithm
ranked_cosine = sorted(range(len(target_texts)), key=lambda i: cosine_scores[i], reverse=True)
ranked_dot = sorted(range(len(target_texts)), key=lambda i: dot_scores[i], reverse=True)
ranked_euclidean = sorted(range(len(target_texts)), key=lambda i: euclidean_scores[i], reverse=True)
ranked_manhattan = sorted(range(len(target_texts)), key=lambda i: manhattan_scores[i], reverse=True)

print("Ranking Order (Target Index from Highest -> Lowest Similarity):")
print(f"  Cosine         : {ranked_cosine}")
print(f"  Cosine Dot     : {ranked_dot}")
print(f"  Euclidean (L2) : {ranked_euclidean}")
print(f"  Manhattan (L1) : {ranked_manhattan}")

max_cos_dot_diff = max(abs(c - d) for c, d in zip(cosine_scores, dot_scores))
print(f"Max |Cosine - Cosine Dot| difference on normalized vectors: {max_cos_dot_diff:.2e}")

print("Score Range Summary across the 12 Targets:")
for name, scores in [
    ("Cosine        ", cosine_scores),
    ("Cosine Dot    ", dot_scores),
    ("Euclidean (L2)", euclidean_scores),
    ("Manhattan (L1)", manhattan_scores),
]:
    non_exact = scores[1:]  # exclude index 0 (exact match = 1.0)
    print(
        f"  {name} -> Exact[0]: {scores[0]:.4f} | "
        f"Best Match[1]: {max(non_exact):.4f} | "
        f"Lowest Match: {min(non_exact):.4f} | "
        f"Spread: {max(non_exact) - min(non_exact):.4f}"
    )


Ranking Order (Target Index from Highest -> Lowest Similarity):
  Cosine         : [0, 1, 2, 11, 10, 3, 4, 6, 9, 5, 7, 8]
  Cosine Dot     : [0, 1, 2, 11, 10, 3, 4, 6, 9, 5, 7, 8]
  Euclidean (L2) : [0, 1, 2, 11, 10, 3, 4, 6, 9, 5, 7, 8]
  Manhattan (L1) : [0, 1, 2, 11, 10, 3, 4, 5, 6, 9, 7, 8]
Max |Cosine - Cosine Dot| difference on normalized vectors: 5.96e-08
Score Range Summary across the 12 Targets:
  Cosine         -> Exact[0]: 1.0000 | Best Match[1]: 0.9438 | Lowest Match: 0.7967 | Spread: 0.1471
  Cosine Dot     -> Exact[0]: 1.0000 | Best Match[1]: 0.9438 | Lowest Match: 0.7967 | Spread: 0.1471
  Euclidean (L2) -> Exact[0]: 1.0000 | Best Match[1]: 0.6784 | Lowest Match: 0.5258 | Spread: 0.1526
  Manhattan (L1) -> Exact[0]: 1.0000 | Best Match[1]: 0.0859 | Lowest Match: 0.0483 | Spread: 0.0377


### Data Analysis Key Findings

- **Exact Equivalence of Cosine and Cosine Dot on Normalized Vectors**: Because `VectorService` L2-normalizes every embedding ($\|\mathbf{v}\|_2 = 1.0$), `cosine` and `dot_product` produce identical scores (`0.00e+00` difference), ranging from `1.0000` (Target `[0]` exact match) and `0.9116` (Target `[1]` semantic synonym) down to `0.6868` (Target `[8]` unrelated pediatric visit), while `dot_product` avoids redundant norm calculations at query time.
- **Order Preservation Between Cosine and Euclidean (L2) on the Unit Hypersphere**: Since $\|\mathbf{q} - \mathbf{t}\|_2 = \sqrt{2 - 2(\mathbf{q} \cdot \mathbf{t})}$ for unit vectors, `euclidean` produces the exact same ranking order `[0, 1, 4, 2, 3, 5, 6, 7, 9, 8]` as `cosine` / `dot_product`, with non-exact scores spanning `0.6271` down to `0.4719` (spread `0.1553`).
- **High-Dimensional Compression in Manhattan (L1) Distance**: Summing absolute differences across 768 dimensions yields large L1 distances ($\|\mathbf{q} - \mathbf{t}\|_1 \approx 13\text{--}25$), compressing `manhattan` scores (`1 / (1 + d)`) into `0.0708` to `0.0386` (spread `0.0322`) and slightly shifting mid/low-rank order (`[6, 5]` and `[8, 9]`).

### Insights or Next Steps

- **Use Cosine Dot-Product for Graph Vector Search**: When storing L2-normalized clinical embeddings in Neo4j HNSW vector indexes, use `dot_product` (or `cosine`) for maximum score discrimination (`0.2248` spread) and fastest inner-product query execution.
- **Experiment with Custom Clinical Queries**: Edit `query_text` and `target_texts` in Section 1 (or register a `vertexai` `gemini-embedding-001` `LLM_Config` with `gosmart-graphite configure llm-config`) and re-run all cells to compare domain sensitivity across models.